In [1]:
import pickle
from pathlib import Path
import pandas as pd

import matplotlib.pyplot as plt
import torch
from neuralhydrology.evaluation import metrics
from neuralhydrology.nh_run import start_run, eval_run
import re
import os 
from neuralhydrology.utils.config import Config
from pathlib import Path

In [2]:
cfe_param_dir = '/Users/danielmckenzie/My-Drive/Research/Spatial_Statistics/data/CFE_Config_Cver_from_Luciana/'
basin_id = '02177000'
cfe_param_file_path = cfe_param_dir + basin_id + '_bmi_config_cfe_pass.txt'
print(cfe_param_file_path)

/Users/danielmckenzie/My-Drive/Research/Spatial_Statistics/data/CFE_Config_Cver_from_Luciana/02177000_bmi_config_cfe_pass.txt


In [3]:
with open(cfe_param_file_path, 'r') as f:
    content = f.read()
    

In [56]:
#pattern = r'([\w.]+)\s*=\s*([\d.+-e]*)'
pattern = r'([\w.]+)\s*=\s*([0-9.eE+-]+(?:,\s*[0-9.eE+-]+)*)'
matches_list = re.findall(pattern, content)
matches = {}
for match in matches_list:
    try:
        matches[match[0]] = float(match[1])
    except:
        matches[match[0]] = [float(x) for x in match[1].split(',')]
print(matches)

{'soil_params.depth': 2.0, 'soil_params.b': 4.0, 'soil_params.satdk': 4.22590909090909e-06, 'soil_params.satpsi': 0.18892282698484852, 'soil_params.slop': 0.3349330841969697, 'soil_params.smcmax': 0.4888061113030302, 'soil_params.wltsmc': 0.05205364346969697, 'soil_params.expon': 1.0, 'soil_params.expon_secondary': 1.0, 'refkdt': 3.6773888333939397, 'max_gw_storage': 0.24879675293000003, 'Cgw': 1.8e-05, 'expon': 2.0, 'gw_storage': 0.05, 'alpha_fc': 0.33, 'soil_storage': 0.05, 'K_nash': 0.03, 'K_lf': 0.01, 'nash_storage': [0.0, 0.0], 'num_timesteps': 1.0, 'verbosity': 1.0, 'DEBUG': 0.0, 'giuh_ordinates': [0.33, 0.29, 0.19, 0.11, 0.05, 0.02, 0.01, 0.0, 0.0, 0.0, 0.0]}


AttributeError: 'list' object has no attribute 'type'

In [57]:
def get_dcfe_params(cfg, device):
    cfe_param_dir = cfg.param_dir
    basin_id = cfg.basin_id
    cfe_param_file_path = cfe_param_dir / (basin_id + '_bmi_config_cfe_pass.txt')
    with open(cfe_param_file_path, 'r') as f:
        content = f.read()
    f.close()
    pattern = r'([\w.]+)\s*=\s*([0-9.eE+-]+(?:,\s*[0-9.eE+-]+)*)'
    matches_list = re.findall(pattern, content)
    matches = {}
    for match in matches_list:
        try:
            matches[match[0]] = float(match[1])
        except:
            matches[match[0]] = [float(x) for x in match[1].split(',')]
    soil_params = {'depth': torch.tensor(matches["soil_params.depth"],device=device,dtype=torch.float32),
                   'bb': torch.tensor(matches["soil_params.b"],device=device,dtype=torch.float32),
                   'satdk': torch.tensor(matches["soil_params.satdk"],device=device,dtype=torch.float32),
                   'satpsi': torch.tensor(matches["soil_params.satpsi"],device=device,dtype=torch.float32),
                   'slop': torch.tensor(matches["soil_params.slop"],device=device,dtype=torch.float32),
                   'smcmax': torch.tensor(matches["soil_params.smcmax"],device=device,dtype=torch.float32),
                   'wltsmc': torch.tensor(matches["soil_params.wltsmc"],device=device,dtype=torch.float32),
                   'D': torch.tensor(2.0, device=device, dtype=torch.float32),
                   'mult': torch.tensor(1.0, device=device, dtype=torch.float32),
                   }
    basinCharacteristics = {'catchment_area_km2': torch.tensor(526.77, device=device, dtype=torch.float32),
                            'refkdt': torch.tensor(matches["refkdt"], device=device, dtype=torch.float32),
                            'max_gw_storage': torch.tensor(matches['max_gw_storage'], device=device, dtype=torch.float32),
                            'expon': torch.tensor(matches['expon'], device=device, dtype=torch.float32),
                            'alpha_fc': torch.tensor(matches['alpha_fc'], device=device, dtype=torch.float32),
                            'K_nash': torch.tensor(matches['K_nash'], device=device, dtype=torch.float32),
                            'K_lf': torch.tensor(matches['K_lf'], device=device, dtype=torch.float32),
                            'nash_storage': torch.tensor(matches['nash_storage'], device=device, dtype=torch.float32),
                            'giuh_ordinates': torch.tensor(matches['giuh_ordinates'], device=device, dtype=torch.float32),
                    }       
    return soil_params, basinCharacteristics

In [58]:
config_path = Path('/Users/danielmckenzie/My-Drive/Research/Spatial_Statistics/neuralhydrology/examples/07-DifferentialCFE-Model/basin_dCFEwPETDaily.yml')
config = Config(config_path, dev_mode=True)

In [59]:
print(config)

In [60]:
get_soil_params(config, 'mps')

({'depth': tensor(2., device='mps:0'),
  'bb': tensor(4., device='mps:0'),
  'satdk': tensor(4.2259e-06, device='mps:0'),
  'satpsi': tensor(0.1889, device='mps:0'),
  'slop': tensor(0.3349, device='mps:0'),
  'smcmax': tensor(0.4888, device='mps:0'),
  'wltsmc': tensor(0.0521, device='mps:0'),
  'D': tensor(2., device='mps:0'),
  'mult': tensor(1., device='mps:0')},
 {'catchment_area_km2': tensor(526.7700, device='mps:0'),
  'refkdt': tensor(3.6774, device='mps:0'),
  'max_gw_storage': tensor(0.2488, device='mps:0'),
  'expon': tensor(2., device='mps:0'),
  'alpha_fc': tensor(0.3300, device='mps:0'),
  'K_nash': tensor(0.0300, device='mps:0'),
  'K_lf': tensor(0.0100, device='mps:0'),
  'nash_storage': tensor([0., 0.], device='mps:0'),
  'giuh_ordinates': tensor([0.3300, 0.2900, 0.1900, 0.1100, 0.0500, 0.0200, 0.0100, 0.0000, 0.0000,
          0.0000, 0.0000], device='mps:0')})

In [61]:
soil_params, basinCharacteristics = get_soil_params(config, 'mps')